# Whisper + PhoWhisper LoRA Fine-tuning
## Vietnamese-English Code-Switching ASR

**Baseline (Whisper large-v3, no fine-tune):**
- WER raw: 25.51% | CER raw: 10.29% | SER raw: 92.76%
- WER norm: 14.91% | CER norm: 7.83% | SER norm: 58.80%

**Pipeline:**
1. Setup & install
2. Configuration
3. Data loading
4. LoRA model setup
5. Training
6. Save adapter

## 1. Setup

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 evaluate==0.4.1 \
    jiwer==3.0.3 datasets==2.19.0 accelerate==0.29.3 \
    soundfile librosa bitsandbytes -q

In [ ]:
import os
import gc
import re
import json
import shutil
import numpy as np
import pandas as pd
import torch
import librosa
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List
from tqdm.notebook import tqdm

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from datasets import Dataset
import evaluate
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, PeftModelForSeq2SeqLM
from jiwer import wer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration
**Chỉ cần chỉnh section này**

In [ ]:
# ── Unzip data (chạy 1 lần) ────────────────────────────────────────────────
!unzip -q train.zip -d train
!unzip -q dev.zip   -d dev
!unzip -q test.zip  -d test

In [ ]:
# ── Data paths ─────────────────────────────────────────────────────────────
DATA_ROOT = "."   # folder chứa train/, dev/, test/

TRAIN_AUDIO_DIR  = os.path.join(DATA_ROOT, "train", "audio")
TRAIN_TRANSCRIPT = os.path.join(DATA_ROOT, "train", "transcript.json")
DEV_AUDIO_DIR    = os.path.join(DATA_ROOT, "dev",   "audio")
DEV_TRANSCRIPT   = os.path.join(DATA_ROOT, "dev",   "transcript.json")
TEST_AUDIO_DIR   = os.path.join(DATA_ROOT, "test",  "audio")
TEST_TRANSCRIPT  = os.path.join(DATA_ROOT, "test",  "transcript.json")

# ── Models to train ────────────────────────────────────────────────────────
# Comment out bất kỳ model nào không muốn train
MODEL_CONFIGS = {
    "whisper-large-v3": {
        "model_id":   "openai/whisper-large-v3",
        "language":   "vi",
        "task":       "transcribe",
        "output_dir": "./output/whisper-large-v3-lora",
        "cache_dir":  "./cache/whisper-large-v3",
    },
    "PhoWhisper-large": {
        "model_id":   "vinai/PhoWhisper-large",
        "language":   "vi",
        "task":       "transcribe",
        "output_dir": "./output/PhoWhisper-large-lora",
        "cache_dir":  "./cache/PhoWhisper-large",
    },
}

# ── LoRA hyperparameters ───────────────────────────────────────────────────
LORA_CONFIG = {
    "r":              32,
    "lora_alpha":     64,
    "lora_dropout":   0.05,
    "target_modules": ["q_proj", "v_proj", "out_proj", "fc1", "fc2"],
    "bias":           "none",
}

# ── Training hyperparameters ───────────────────────────────────────────────
TRAINING_CONFIG = {
    "num_train_epochs":             5,
    "per_device_train_batch_size":  8,
    "per_device_eval_batch_size":   8,
    "gradient_accumulation_steps":  2,
    "learning_rate":                1e-4,
    "warmup_steps":                 100,
    "fp16":                         False,
    "bf16":                         True,
    "evaluation_strategy":          "epoch",
    "save_strategy":                "epoch",
    "load_best_model_at_end":       True,
    "metric_for_best_model":        "wer",
    "greater_is_better":            False,
    "predict_with_generate":        True,
    "generation_max_length":        225,
    "logging_steps":                25,
    "dataloader_num_workers":       4,
    "weight_decay":                 0.01,
    "early_stopping_patience":      3,
    "gradient_checkpointing":       False,
}

SAMPLING_RATE     = 16000
MAX_AUDIO_LEN_SEC = 30

BASELINE_METRICS = {
    "model":    "Whisper large-v3 (baseline)",
    "wer_raw":  25.51, "cer_raw":  10.29, "ser_raw":  92.76,
    "wer_norm": 14.91, "cer_norm":  7.83, "ser_norm": 58.80,
}

# WandB Training track
os.environ["WANDB_DISABLED"] = "true"

print("Configuration loaded.")

## 3. Data Loading

In [ ]:
def load_audio(path: str, target_sr: int = SAMPLING_RATE) -> np.ndarray:
    audio, _ = librosa.load(path, sr=target_sr, mono=True)
    max_samples = MAX_AUDIO_LEN_SEC * target_sr
    return audio[:max_samples] if len(audio) > max_samples else audio


def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_dataset_from_json(audio_dir: str, transcript_path: str) -> List[Dict]:
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcripts = json.load(f)
    samples, missing = [], 0
    for filename, meta in transcripts.items():
        path = os.path.join(audio_dir, filename)
        if os.path.exists(path):
            samples.append({"audio_path": path, "filename": filename, "text": meta["text"]})
        else:
            missing += 1
    print(f"  Loaded {len(samples)} samples (skipped {missing} missing)")
    return samples


def prepare_hf_dataset(samples, processor, language, task):
    processor.tokenizer.set_prefix_tokens(language=language, task=task)

    def preprocess(batch):
        audio = load_audio(batch["audio_path"])
        batch["input_features"] = processor.feature_extractor(
            audio, sampling_rate=SAMPLING_RATE
        ).input_features[0]
        batch["labels"] = processor.tokenizer(batch["text"]).input_ids
        return batch

    hf_dataset = Dataset.from_list(samples)
    hf_dataset = hf_dataset.map(preprocess, desc="Preprocessing audio", num_proc=1)
    hf_dataset = hf_dataset.remove_columns(["audio_path", "filename"])
    return hf_dataset


print("Loading datasets...")
train_samples = load_dataset_from_json(TRAIN_AUDIO_DIR, TRAIN_TRANSCRIPT)
dev_samples   = load_dataset_from_json(DEV_AUDIO_DIR,   DEV_TRANSCRIPT)
test_samples  = load_dataset_from_json(TEST_AUDIO_DIR,  TEST_TRANSCRIPT)

print(f"\nTrain: {len(train_samples)} | Dev: {len(dev_samples)} | Test: {len(test_samples)}")

## 4. LoRA Model & Training Utilities

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        batch.pop("input_ids", None)
        return batch


def build_lora_model(model_id: str):
    print(f"Loading model: {model_id}")
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float32, device_map="auto"
    )
    model.config.use_cache = False
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        target_modules=LORA_CONFIG["target_modules"],
        bias=LORA_CONFIG["bias"],
    )
    model = get_peft_model(model, lora_config)
    model.enable_input_require_grads()
    model.base_model.model.model.encoder.gradient_checkpointing = False
    model.print_trainable_parameters()

    # Fix: patch PEFT forward for Whisper (input_features vs input_ids)
    original_forward = PeftModelForSeq2SeqLM.forward

    def patched_forward(self, input_ids=None, attention_mask=None,
                        inputs_embeds=None, decoder_input_ids=None,
                        decoder_attention_mask=None, decoder_inputs_embeds=None,
                        labels=None, output_attentions=None,
                        output_hidden_states=None, return_dict=None,
                        task_ids=None, **kwargs):
        if "input_features" in kwargs:
            input_features = kwargs.pop("input_features")
            with self._enable_peft_forward_hooks(**kwargs):
                kwargs = {k: v for k, v in kwargs.items()
                          if k not in self.special_peft_forward_args}
                return self.base_model(
                    input_features=input_features,
                    attention_mask=attention_mask,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=decoder_attention_mask,
                    decoder_inputs_embeds=decoder_inputs_embeds,
                    labels=labels,
                    output_attentions=output_attentions,
                    output_hidden_states=output_hidden_states,
                    return_dict=return_dict,
                    **kwargs,
                )
        return original_forward(
            self, input_ids=input_ids, attention_mask=attention_mask,
            inputs_embeds=inputs_embeds, decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds, labels=labels,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict, task_ids=task_ids, **kwargs,
        )

    PeftModelForSeq2SeqLM.forward = patched_forward
    print("PEFT forward patched.")
    return model, processor


wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_ser(references, hypotheses):
    errors = sum(r.strip() != h.strip() for r, h in zip(references, hypotheses))
    return (errors / len(references)) * 100


def compute_all_metrics(references, hypotheses):
    wer_raw = wer_metric.compute(references=references, predictions=hypotheses) * 100
    cer_raw = cer_metric.compute(references=references, predictions=hypotheses) * 100
    ser_raw = compute_ser(references, hypotheses)
    refs_n = [normalize_text(r) for r in references]
    hyps_n = [normalize_text(h) for h in hypotheses]
    wer_norm = wer_metric.compute(references=refs_n, predictions=hyps_n) * 100
    cer_norm = cer_metric.compute(references=refs_n, predictions=hyps_n) * 100
    ser_norm = compute_ser(refs_n, hyps_n)
    return {
        "wer_raw": round(wer_raw, 2), "cer_raw": round(cer_raw, 2), "ser_raw": round(ser_raw, 2),
        "wer_norm": round(wer_norm, 2), "cer_norm": round(cer_norm, 2), "ser_norm": round(ser_norm, 2),
    }


def train_model(model_name, config, train_samples, dev_samples, resume_from_checkpoint=None):
    output_dir = config["output_dir"]
    cache_dir  = config["cache_dir"]
    os.makedirs(cache_dir, exist_ok=True)

    model, processor = build_lora_model(config["model_id"])

    # Prepare datasets (cache to disk to avoid reprocessing)
    train_cache = os.path.join(cache_dir, "train")
    dev_cache   = os.path.join(cache_dir, "dev")

    if os.path.exists(train_cache) and os.path.exists(dev_cache):
        from datasets import load_from_disk
        print("Loading cached datasets...")
        train_dataset = load_from_disk(train_cache)
        dev_dataset   = load_from_disk(dev_cache)
    else:
        print("Preprocessing datasets...")
        train_dataset = prepare_hf_dataset(train_samples, processor, config["language"], config["task"])
        dev_dataset   = prepare_hf_dataset(dev_samples,   processor, config["language"], config["task"])
        train_dataset.save_to_disk(train_cache)
        dev_dataset.save_to_disk(dev_cache)

    print(f"Train: {len(train_dataset)} | Dev: {len(dev_dataset)}")

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        wer_score = wer_metric.compute(references=label_str, predictions=pred_str) * 100
        return {"wer": round(wer_score, 2)}

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
        gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        warmup_steps=TRAINING_CONFIG["warmup_steps"],
        fp16=TRAINING_CONFIG["fp16"],
        bf16=TRAINING_CONFIG["bf16"],
        evaluation_strategy=TRAINING_CONFIG["evaluation_strategy"],
        save_strategy=TRAINING_CONFIG["save_strategy"],
        load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
        metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
        greater_is_better=TRAINING_CONFIG["greater_is_better"],
        predict_with_generate=TRAINING_CONFIG["predict_with_generate"],
        generation_max_length=TRAINING_CONFIG["generation_max_length"],
        logging_steps=TRAINING_CONFIG["logging_steps"],
        dataloader_num_workers=TRAINING_CONFIG["dataloader_num_workers"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
        gradient_checkpointing=TRAINING_CONFIG["gradient_checkpointing"],
        remove_unused_columns=False,
        label_names=["labels"],
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=TRAINING_CONFIG["early_stopping_patience"]
        )],
    )

    if resume_from_checkpoint:
        print(f"Resuming from: {resume_from_checkpoint}")
    else:
        print("Training from scratch...")

    trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    print(f"\nTraining complete: {model_name}")
    return model, processor, output_dir


print("Utilities defined.")

## 5. Training
Chạy từng cell độc lập — không ảnh hưởng nhau.

In [ ]:
# ── 5a. Train Whisper large-v3 ─────────────────────────────────────────────
def _auto_resume(output_dir):
    if not os.path.exists(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    print(f"Resuming from: {latest}")
    return os.path.join(output_dir, latest)


config = MODEL_CONFIGS["whisper-large-v3"]
model_whisper, processor_whisper, _ = train_model(
    "whisper-large-v3", config, train_samples, dev_samples,
    resume_from_checkpoint=_auto_resume(config["output_dir"]),
)
del model_whisper
torch.cuda.empty_cache()
gc.collect()
print("✅ Whisper large-v3 done")

In [ ]:
# ── 5b. Train PhoWhisper-large ─────────────────────────────────────────────
config = MODEL_CONFIGS["PhoWhisper-large"]
model_pho, processor_pho, _ = train_model(
    "PhoWhisper-large", config, train_samples, dev_samples,
    resume_from_checkpoint=_auto_resume(config["output_dir"]),
)
del model_pho
torch.cuda.empty_cache()
gc.collect()
print("✅ PhoWhisper-large done")

## 6. Save Adapters

In [ ]:
# Zip adapters để download
SAVE_DIR = "./saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

adapter_dirs = {
    "whisper-large-v3-lora-adapter": "./output/whisper-large-v3-lora",
    "PhoWhisper-large-lora-adapter":  "./output/PhoWhisper-large-lora",
}

for zip_name, adapter_dir in adapter_dirs.items():
    if not os.path.exists(adapter_dir):
        print(f"Not found: {adapter_dir} — skip")
        continue
    zip_path = os.path.join(SAVE_DIR, zip_name)
    shutil.make_archive(zip_path, "zip", adapter_dir)
    size_mb = os.path.getsize(zip_path + ".zip") / 1e6
    print(f"✅ {zip_name}.zip — {size_mb:.1f} MB")

print(f"\nDone! Download from: {SAVE_DIR}/")